In [1]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, count, sum as _sum, year
import requests
import pandas as pd


pg_url = "jdbc:postgresql://metastore-db:5432/hive_metastore?createDatabaseIfNotExist=false&sslmode=disable"
pg_user = "hive"
pg_pass = "hivepass"

print("Iniciando Spark: ")

spark = SparkSession.builder \
    .appName("SECOP_ETL") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL", pg_url) \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName", "org.postgresql.Driver") \
    .config("spark.hadoop.javax.jdo.option.ConnectionUserName", pg_user) \
    .config("spark.hadoop.javax.jdo.option.ConnectionPassword", pg_pass) \
    .config("spark.hadoop.datanucleus.schema.autoCreateAll", "false") \
    .config("spark.hadoop.hive.metastore.schema.verification", "false") \
    .enableHiveSupport() \
    .getOrCreate()

print(f"✅ Spark Listo. Versión: {spark.version}")

Iniciando Spark: 


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/14 06:03:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark Listo. Versión: 3.5.0


In [2]:
print("🕵️‍♀️ Validando conexión al Metastore...")
try:
    spark.sql("CREATE DATABASE IF NOT EXISTS test_connection")
    spark.sql("SHOW DATABASES").show()
    spark.sql("DROP DATABASE test_connection")
    spark.sql("DROP DATABASE secop")
    print("🎉 CONEXIÓN EXITOSA: El Metastore responde y la configuración es correcta.")
except Exception as e:
    print(f"❌ ERROR CRÍTICO: {e}")

🕵️‍♀️ Validando conexión al Metastore...


25/12/13 03:42:23 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/12/13 03:42:23 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
25/12/13 03:42:23 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
25/12/13 03:42:23 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@10.89.0.7
25/12/13 03:42:23 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException
25/12/13 03:42:24 WARN ObjectStore: Failed to get database test_connection, returning NoSuchObjectException
25/12/13 03:42:24 WARN ObjectStore: Failed to get database test_connection, returning NoSuchObjectException
25/12/13 03:42:24 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
25/12/13 03:42:24 WARN ObjectStore: Failed to get database test_conn

+---------------+
|      namespace|
+---------------+
|        default|
|test_connection|
+---------------+

❌ ERROR CRÍTICO: [SCHEMA_NOT_FOUND] The schema `secop` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a catalog, verify the current_schema() output, or qualify the name with the correct catalog.
To tolerate the error on drop use DROP SCHEMA IF EXISTS.


25/12/13 03:42:25 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
25/12/13 03:42:25 WARN ObjectStore: Failed to get database secop, returning NoSuchObjectException


In [3]:
# Configuración de la API
BASE_URL = "https://www.datos.gov.co/resource/jbjy-vk9h.json"
LIMIT = 5000       # Tamaño del lote
Total_LIMIT = 20000 # Total a descargar (puedes subirlo si quieres)
offset = 0

# Ruta temporal en HDFS donde aterrizan los datos crudos
landing_path = "hdfs://namenode:9000/user/jovyan/secop_landing_json"

print(f"⬇️ Iniciando descarga hacia HDFS: {landing_path}")

# Limpiamos la zona de aterrizaje si ya existía (para empezar de cero)
# Usamos el FileSystem de Hadoop a través de la JVM de Spark
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
fs.delete(spark._jvm.org.apache.hadoop.fs.Path(landing_path), True)

while offset < Total_LIMIT:
    url = f"{BASE_URL}?$limit={LIMIT}&$offset={offset}"
    try:
        r = requests.get(url, timeout=20)
        batch = r.json()
    except Exception as e:
        print(f"❌ Error descargando offset {offset}: {e}")
        break

    if not batch:
        break

    # Truco Pro: Convertimos a Pandas -> String para evitar errores de esquema inferido
    # y luego pasamos a Spark para escribir en disco inmediatamente.
    pdf_small = pd.DataFrame(batch).astype(str)
    df_batch = spark.createDataFrame(pdf_small)
    
    # Escribimos en modo "append" (agregar al final)
    df_batch.write.mode("append").json(landing_path)
    
    print(f"  - Lote offset={offset} guardado en HDFS. RAM liberada.")
    
    # Limpieza manual de variables Python para asegurar memoria libre
    del batch, pdf_small, df_batch
    offset += LIMIT

print("✅ Ingesta finalizada. Los datos están seguros en HDFS.")

⬇️ Iniciando descarga hacia HDFS: hdfs://namenode:9000/user/jovyan/secop_landing_json


25/12/13 03:42:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/12/13 03:42:38 WARN TaskSetManager: Stage 0 contains a task of very large size (1086 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  - Lote offset=0 guardado en HDFS. RAM liberada.


25/12/13 03:42:47 WARN TaskSetManager: Stage 1 contains a task of very large size (1086 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  - Lote offset=5000 guardado en HDFS. RAM liberada.


25/12/13 03:42:53 WARN TaskSetManager: Stage 2 contains a task of very large size (1085 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  - Lote offset=10000 guardado en HDFS. RAM liberada.


25/12/13 03:42:59 WARN TaskSetManager: Stage 3 contains a task of very large size (1081 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  - Lote offset=15000 guardado en HDFS. RAM liberada.
✅ Ingesta finalizada. Los datos están seguros en HDFS.


In [4]:
# Definimos la ruta donde guardamos los datos
path = "hdfs://namenode:9000/user/jovyan/secop_landing_json"

print(f"🕵️‍♀️ Inspeccionando ruta: {path}")

try:
    # Intentamos leer lo que acabamos de guardar
    df_check = spark.read.json(path)
    
    # Contamos registros (esto obliga a Spark a recorrer los archivos)
    total = df_check.count()
    print(f"✅ ¡Confirmado! Hay {total} registros accesibles en HDFS.")
    
    # Mostramos la estructura de archivos física
    # (Usamos la JVM de Hadoop para listar los archivos "part-xxxxx")
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
    list_status = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path(path))
    
    print("\n📂 Archivos físicos encontrados:")
    for file in list_status:
        print(f" - {file.getPath().getName()} ({file.getLen()} bytes)")
        
except Exception as e:
    print(f"❌ Error leyendo HDFS: {e}")

🕵️‍♀️ Inspeccionando ruta: hdfs://namenode:9000/user/jovyan/secop_landing_json


[Stage 5:=======>                                                   (1 + 7) / 8]

✅ ¡Confirmado! Hay 20000 registros accesibles en HDFS.

📂 Archivos físicos encontrados:
 - _SUCCESS (0 bytes)
 - part-00000-222477e8-cb48-4225-846b-ecdf748ab20c-c000.json (2401796 bytes)
 - part-00000-4050c169-f3e2-4229-91ec-587865bb01b6-c000.json (2401802 bytes)
 - part-00000-93dbca22-3c8d-4896-8f84-cdfc2f5f5956-c000.json (2396626 bytes)
 - part-00000-deb9f072-dbad-427f-9d21-6f9cc2eab642-c000.json (2400949 bytes)
 - part-00001-222477e8-cb48-4225-846b-ecdf748ab20c-c000.json (2397658 bytes)
 - part-00001-4050c169-f3e2-4229-91ec-587865bb01b6-c000.json (2398534 bytes)
 - part-00001-93dbca22-3c8d-4896-8f84-cdfc2f5f5956-c000.json (2386096 bytes)
 - part-00001-deb9f072-dbad-427f-9d21-6f9cc2eab642-c000.json (2404561 bytes)
 - part-00002-222477e8-cb48-4225-846b-ecdf748ab20c-c000.json (2397779 bytes)
 - part-00002-4050c169-f3e2-4229-91ec-587865bb01b6-c000.json (2395856 bytes)
 - part-00002-93dbca22-3c8d-4896-8f84-cdfc2f5f5956-c000.json (2401371 bytes)
 - part-00002-deb9f072-dbad-427f-9d21-6f9cc

In [5]:
from pyspark.sql.functions import col, to_date, count, sum as _sum, year

print("📂 Leyendo datos crudos desde HDFS...")
df_raw = spark.read.json("hdfs://namenode:9000/user/jovyan/secop_landing_json")

# Estrategia Simplificada: No hacemos explode. Pasamos directo a limpiar.
df_flat = df_raw

print("  - Aplicando reglas de negocio y tipos de datos...")
df_clean = df_flat.select(
    col("id_contrato").alias("idcontrato"),
    col("estado_contrato"),
    col("modalidad_de_contratacion").alias("modalidad_contratacion"),
    col("entidad_centralizada"),
    col("departamento"),
    # Convertimos string a fecha real
    to_date("fecha_de_firma").alias("fecha_firma"),
    # Convertimos string a double (moneda)
    col("valor_del_contrato").cast("double").alias("valor_contrato")
).na.drop(subset=["idcontrato", "valor_contrato", "fecha_firma"])

print("📊 Esquema Final Definido:")
df_clean.printSchema()

print(f"✅ Registros limpios listos para guardar: {df_clean.count()}")
print("Muestra de datos:")
df_clean.show(5)

📂 Leyendo datos crudos desde HDFS...
  - Aplicando reglas de negocio y tipos de datos...
📊 Esquema Final Definido:
root
 |-- idcontrato: string (nullable = true)
 |-- estado_contrato: string (nullable = true)
 |-- modalidad_contratacion: string (nullable = true)
 |-- entidad_centralizada: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- fecha_firma: date (nullable = true)
 |-- valor_contrato: double (nullable = true)



✅ Registros limpios listos para guardar: 18440
Muestra de datos:
+------------------+---------------+----------------------+--------------------+--------------------+-----------+--------------+
|        idcontrato|estado_contrato|modalidad_contratacion|entidad_centralizada|        departamento|fecha_firma|valor_contrato|
+------------------+---------------+----------------------+--------------------+--------------------+-----------+--------------+
|CO1.PCCNTR.2252730|      terminado|  Contratación directa|        Centralizada|           Santander| 2021-02-13|        1.62E7|
|CO1.PCCNTR.3506382|      terminado|  Contratación directa|        Centralizada|           Santander| 2022-01-28|     7600000.0|
|CO1.PCCNTR.1869780|       Aprobado|  Contratación régi...|        Centralizada|Distrito Capital ...| 2020-10-01|     5253309.0|
| CO1.PCCNTR.306701|     Modificado|  Contratación directa|        Centralizada|           Antioquia| 2018-01-22|   1.4246136E7|
|CO1.PCCNTR.4783324|     Modific

In [6]:
print("📖 Gestionando Metastore (Postgres)...")

try:
    # 1. Crear la base de datos lógica
    spark.sql("CREATE DATABASE IF NOT EXISTS secop")
    print("   - Base de datos 'secop' verificada.")

    print("💾 Guardando tabla maestra 'secop.contratos' (Parquet + Hive)...")
    
    # 2. Escritura Física y Registro de Metadatos
    # mode("overwrite") es vital aquí por si re-ejecutas el notebook
    df_clean.write \
        .mode("overwrite") \
        .format("parquet") \
        .saveAsTable("secop.contratos")

    print("✅ ¡VICTORIA! Tabla registrada exitosamente en el Metastore.")

    # --- ANÁLISIS DE NEGOCIO (Prueba de Fuego) ---
    # Si esta consulta corre, significa que:
    # A. Los datos están en disco (HDFS).
    # B. El Metastore sabe dónde están y qué columnas tienen (Postgres).
    # C. Spark puede leerlos y procesarlos.
    
    print("\n🔍 TOP 5 Departamentos con mayor dinero contratado:")
    spark.sql("""
        SELECT 
            departamento,
            year(fecha_firma) as anio, 
            count(*) as cantidad_contratos, 
            concat('$', format_number(sum(valor_contrato), 0)) as dinero_total
        FROM secop.contratos 
        WHERE valor_contrato > 0
        GROUP BY 1, 2
        ORDER BY sum(valor_contrato) DESC
        LIMIT 5
    """).show(truncate=False)

except Exception as e:
    print("\n❌ FALLO EN LA PERSISTENCIA FINAL:")
    print(f"Error: {e}")

📖 Gestionando Metastore (Postgres)...
   - Base de datos 'secop' verificada.
💾 Guardando tabla maestra 'secop.contratos' (Parquet + Hive)...


25/12/13 03:43:19 WARN ObjectStore: Failed to get database secop, returning NoSuchObjectException
25/12/13 03:43:19 WARN ObjectStore: Failed to get database secop, returning NoSuchObjectException
25/12/13 03:43:19 WARN ObjectStore: Failed to get database secop, returning NoSuchObjectException
25/12/13 03:43:21 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
25/12/13 03:43:21 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
25/12/13 03:43:21 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/12/13 03:43:21 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


✅ ¡VICTORIA! Tabla registrada exitosamente en el Metastore.

🔍 TOP 5 Departamentos con mayor dinero contratado:


[Stage 14:>                                                         (0 + 8) / 8]

+--------------------------+----+------------------+----------------+
|departamento              |anio|cantidad_contratos|dinero_total    |
+--------------------------+----+------------------+----------------+
|Antioquia                 |2021|321               |$461,713,995,859|
|Distrito Capital de Bogotá|2022|1534              |$317,394,411,845|
|Distrito Capital de Bogotá|2021|1495              |$287,773,719,043|
|Distrito Capital de Bogotá|2023|1692              |$229,420,390,973|
|Distrito Capital de Bogotá|2020|1316              |$156,227,785,561|
+--------------------------+----+------------------+----------------+



In [7]:
print("\n🔍 TOP 5 Departamentos con mayor contratación:")
spark.sql("""
    SELECT 
        departamento,
        year(fecha_firma) as anio, 
        count(*) as cantidad_contratos, 
        concat('$', format_number(sum(valor_contrato), 0)) as dinero_total
    FROM secop.contratos 
    WHERE valor_contrato > 0
    GROUP BY 1, 2
    ORDER BY sum(valor_contrato) DESC
    LIMIT 5
""").show(truncate=False)


🔍 TOP 5 Departamentos con mayor contratación:
+--------------------------+----+------------------+----------------+
|departamento              |anio|cantidad_contratos|dinero_total    |
+--------------------------+----+------------------+----------------+
|Antioquia                 |2021|321               |$461,713,995,859|
|Distrito Capital de Bogotá|2022|1534              |$317,394,411,845|
|Distrito Capital de Bogotá|2021|1495              |$287,773,719,043|
|Distrito Capital de Bogotá|2023|1692              |$229,420,390,973|
|Distrito Capital de Bogotá|2020|1316              |$156,227,785,561|
+--------------------------+----+------------------+----------------+



In [5]:
print("🗑️ Ejecutando limpieza profunda del Metastore y HDFS...")

# DROP DATABASE ... CASCADE borra la base de datos, todas las tablas dentro
# Y, CRUCIALMENTE, elimina la carpeta de datos asociada en HDFS.
spark.sql("DROP DATABASE IF EXISTS secop CASCADE")

print("✅ Base de datos 'secop' eliminada del Metastore y HDFS.")
print("Ahora la ruta 'hdfs://namenode:9000/user/hive/warehouse/secop.db' está vacía.")

🗑️ Ejecutando limpieza profunda del Metastore y HDFS...


25/12/12 06:42:23 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/12/12 06:42:23 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
25/12/12 06:42:24 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
25/12/12 06:42:24 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@10.89.0.7
25/12/12 06:42:24 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


✅ Base de datos 'secop' eliminada del Metastore y HDFS.
Ahora la ruta 'hdfs://namenode:9000/user/hive/warehouse/secop.db' está vacía.


25/12/12 06:42:25 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
